# Step 3 V2 — DPO Alignment: β Sweep + Epoch Sweep

**NeurIPS additions over V1:**
- Applies DPO to each method's output from Step 2 (not just fedavg_proj)
- β sweep: {0.01, 0.05, 0.1, 0.5}
- Epoch sweep: {1, 3, 5, 10}
- Multi-seed
- Tracks per-epoch DPO loss for convergence curves

**Produces:**
- `models/step3_{method}_seed{s}/` — DPO-aligned LoRA checkpoints
- `results/step3_dpo_results.pt` — per-run metrics
- `results/step3_beta_sweep.pt` — β ablation
- `plots/step3_dpo_loss_curves.png`

In [ ]:
import sys
print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'Python: {sys.version}')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)')
print('=' * 60)

In [ ]:
%pip install -q transformers datasets peft evaluate matplotlib

In [ ]:
import sys
sys.path.insert(0, '.')
from utils import *
import matplotlib.pyplot as plt
from peft import PeftModel
from tqdm.auto import tqdm

# ── Step 3 config ──────────────────────────────────────────────────────────────
METHODS_TO_ALIGN = ['fedavg_proj', 'fedavg_noproj', 'fedprox_proj']
BETA_SWEEP_VALUES = [0.01, 0.05, 0.1, 0.5]
EPOCH_SWEEP_VALUES = [1, 3, 5, 10]

MAIN_BETA   = 0.05   # default for main results
MAIN_EPOCHS = 5

GPU_ID  = 1 if torch.cuda.device_count() > 1 else 0
_DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

print(f'Methods to align : {METHODS_TO_ALIGN}')
print(f'Beta sweep       : {BETA_SWEEP_VALUES}')
print(f'Epoch sweep      : {EPOCH_SWEEP_VALUES}')
print(f'Device           : {_DEVICE}')

In [ ]:
def run_dpo(
    base_lora_dir,          # PeftModel base (safety LoRA)
    agg_weights_path,       # Step 2 output LoRA weights (state dict)
    dpo_pairs,
    tokenizer,
    output_dir: Path,
    beta: float = 0.05,
    n_epochs: int = 5,
    lr: float = LR_DPO,
    device=_DEVICE,
) -> List[float]:
    """
    Run DPO alignment. Returns per-epoch loss list.
    """
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    torch.cuda.set_device(GPU_ID)

    # Policy model
    base_p, _ = load_base_model_and_tokenizer(gpu_id=GPU_ID)
    policy = PeftModel.from_pretrained(base_p, str(base_lora_dir), is_trainable=True)
    if agg_weights_path is not None and Path(str(agg_weights_path)).exists():
        agg = torch.load(str(agg_weights_path), map_location='cpu')
        sd = policy.state_dict()
        for k, v in agg.items():
            if k in sd:
                sd[k] = v.to(dtype=DTYPE)
        policy.load_state_dict(sd, strict=False)

    policy.gradient_checkpointing_enable()
    policy.enable_input_require_grads()

    # Reference model (frozen safety LoRA)
    base_r, _ = load_base_model_and_tokenizer(gpu_id=GPU_ID)
    ref = PeftModel.from_pretrained(base_r, str(base_lora_dir), is_trainable=False)
    for p in ref.parameters():
        p.requires_grad_(False)
    ref.eval()

    chosen_texts   = [p[0] for p in dpo_pairs]
    rejected_texts = [p[1] for p in dpo_pairs]

    chosen_enc = tokenizer(chosen_texts, truncation=True, max_length=MAX_SEQ_LEN,
                           padding='max_length', return_tensors='pt')
    rej_enc    = tokenizer(rejected_texts, truncation=True, max_length=MAX_SEQ_LEN,
                           padding='max_length', return_tensors='pt')

    optimizer = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad], lr=lr)

    epoch_losses = []
    policy.train()

    for epoch in range(n_epochs):
        perm = torch.randperm(len(dpo_pairs))
        total_loss, nb = 0.0, 0
        pbar = tqdm(range(len(dpo_pairs)), desc=f'Epoch {epoch+1}/{n_epochs}')

        for i in pbar:
            idx = perm[i: i+1]
            c_ids  = chosen_enc['input_ids'][idx].to(device)
            c_mask = chosen_enc['attention_mask'][idx].to(device)
            r_ids  = rej_enc['input_ids'][idx].to(device)
            r_mask = rej_enc['attention_mask'][idx].to(device)

            p_c = compute_log_probs(policy, c_ids, c_mask)
            p_r = compute_log_probs(policy, r_ids, r_mask)
            with torch.no_grad():
                ref_c = compute_log_probs(ref, c_ids, c_mask)
                ref_r = compute_log_probs(ref, r_ids, r_mask)

            loss = dpo_loss(p_c, p_r, ref_c, ref_r, beta=beta)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            nb += 1
            pbar.set_postfix({'loss': f'{total_loss/nb:.4f}'})

        epoch_loss = total_loss / max(nb, 1)
        epoch_losses.append(epoch_loss)
        print(f'  Epoch {epoch+1}: DPO loss = {epoch_loss:.4f}')

    output_dir.mkdir(parents=True, exist_ok=True)
    policy.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print(f'  Saved to {output_dir}')

    del base_p, base_r, policy, ref
    torch.cuda.empty_cache()
    return epoch_losses

print('run_dpo defined.')

In [ ]:
# ── Main DPO: apply to each method × each seed ────────────────────────────────
dpo_results_path = RESULTS_DIR / 'step3_dpo_results.pt'

if dpo_results_path.exists():
    dpo_results = torch.load(str(dpo_results_path))
    print(f'Loaded existing DPO results ({len(dpo_results)} entries).')
else:
    dpo_results = []

done_dpo = {(r['method'], r['seed']) for r in dpo_results}

_, tok = load_base_model_and_tokenizer(gpu_id=GPU_ID)

for seed in SEEDS:
    safety_lora = MODELS_DIR / f'safety_lora_seed{seed}'
    dpo_pairs_path = DATA_DIR / f'dpo_pairs_seed{seed}.pt'
    if not dpo_pairs_path.exists():
        print(f'DPO pairs missing for seed {seed}, skipping.')
        continue
    dpo_pairs = torch.load(str(dpo_pairs_path))

    for method in METHODS_TO_ALIGN:
        if (method, seed) in done_dpo:
            print(f'  {method} seed={seed}: already done.')
            continue

        agg_weights = MODELS_DIR / f'step2_{method}_seed{seed}.pt'
        if not agg_weights.exists():
            print(f'  Step 2 weights missing: {agg_weights.name}, skipping.')
            continue

        out_dir = MODELS_DIR / f'step3_{method}_seed{seed}'
        print(f'\n  DPO: {method} seed={seed}')

        epoch_losses = run_dpo(
            base_lora_dir=safety_lora,
            agg_weights_path=agg_weights,
            dpo_pairs=dpo_pairs,
            tokenizer=tok,
            output_dir=out_dir,
            beta=MAIN_BETA,
            n_epochs=MAIN_EPOCHS,
        )

        dpo_results.append({
            'method': method, 'seed': seed,
            'beta': MAIN_BETA, 'n_epochs': MAIN_EPOCHS,
            'epoch_losses': epoch_losses,
            'final_loss': epoch_losses[-1],
            'output_dir': str(out_dir),
        })
        done_dpo.add((method, seed))
        torch.save(dpo_results, str(dpo_results_path))

print(f'\nMain DPO complete: {len(dpo_results)} runs.')

In [ ]:
# ── β sweep on fedavg_proj, 3 seeds ───────────────────────────────────────────
beta_sweep_path = RESULTS_DIR / 'step3_beta_sweep.pt'

if beta_sweep_path.exists():
    beta_sweep = torch.load(str(beta_sweep_path))
    print(f'Loaded beta sweep ({len(beta_sweep)} entries).')
else:
    beta_sweep = []

done_beta = {(r['beta'], r['seed']) for r in beta_sweep}
_, tok = load_base_model_and_tokenizer(gpu_id=GPU_ID)

for seed in SEEDS[:3]:
    safety_lora = MODELS_DIR / f'safety_lora_seed{seed}'
    dpo_pairs   = torch.load(str(DATA_DIR / f'dpo_pairs_seed{seed}.pt'))
    agg_weights = MODELS_DIR / f'step2_fedavg_proj_seed{seed}.pt'
    if not agg_weights.exists():
        continue

    for beta in BETA_SWEEP_VALUES:
        if (beta, seed) in done_beta:
            print(f'  beta={beta} seed={seed}: skip')
            continue

        out_dir = MODELS_DIR / f'step3_beta{beta}_seed{seed}'
        print(f'  beta={beta} seed={seed}')
        epoch_losses = run_dpo(
            base_lora_dir=safety_lora, agg_weights_path=agg_weights,
            dpo_pairs=dpo_pairs, tokenizer=tok, output_dir=out_dir,
            beta=beta, n_epochs=3,
        )
        beta_sweep.append({'beta': beta, 'seed': seed, 'epoch_losses': epoch_losses,
                           'final_loss': epoch_losses[-1]})
        done_beta.add((beta, seed))
        torch.save(beta_sweep, str(beta_sweep_path))

print('Beta sweep complete.')

In [ ]:
# ── Plot DPO loss curves ───────────────────────────────────────────────────────
dpo_results = torch.load(str(RESULTS_DIR / 'step3_dpo_results.pt'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'fedavg_proj': '#2980b9', 'fedavg_noproj': '#e74c3c', 'fedprox_proj': '#27ae60'}

# Left: DPO loss curves per method (seed=42)
for r in dpo_results:
    if r['seed'] != 42:
        continue
    epochs = list(range(1, len(r['epoch_losses']) + 1))
    axes[0].plot(epochs, r['epoch_losses'],
                 label=r['method'], color=colors.get(r['method'], 'gray'),
                 linewidth=2, marker='o', markersize=6)
axes[0].axhline(y=0.693, color='gray', linestyle=':', linewidth=1, label='random (0.693)')
axes[0].set_xlabel('DPO Epoch', fontsize=12)
axes[0].set_ylabel('DPO Loss', fontsize=12)
axes[0].set_title('DPO Convergence by Method (seed=42)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.4)

# Right: β sweep final loss
beta_data = torch.load(str(RESULTS_DIR / 'step3_beta_sweep.pt'))
beta_vals = BETA_SWEEP_VALUES
beta_means = []
beta_stds  = []
for b in beta_vals:
    vals = [r['final_loss'] for r in beta_data if r['beta'] == b]
    beta_means.append(np.mean(vals) if vals else 0)
    beta_stds.append(np.std(vals) if vals else 0)
axes[1].errorbar(beta_vals, beta_means, yerr=beta_stds,
                 fmt='o-', color='steelblue', linewidth=2, markersize=8, capsize=5)
axes[1].axvline(x=0.05, color='red', linestyle='--', linewidth=1.5, label='β=0.05 (default)')
axes[1].set_xscale('log')
axes[1].set_xlabel('β (DPO temperature)', fontsize=12)
axes[1].set_ylabel('Final DPO Loss (3 seeds ± std)', fontsize=12)
axes[1].set_title('DPO β Ablation', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'step3_dpo_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: step3_dpo_curves.png')

In [ ]:
dpo_results = torch.load(str(RESULTS_DIR / 'step3_dpo_results.pt'))
print('\n' + '='*55)
print(f'{"Method":<20} {"Seed":>6} {"Final DPO Loss":>15}')
print('-'*55)
for r in sorted(dpo_results, key=lambda x: (x['method'], x['seed'])):
    print(f"{r['method']:<20} {r['seed']:>6} {r['final_loss']:>15.4f}")
print('='*55)